# 🚀 AppleSupport Chat Ingestion Pipeline for Google Colab & Pinecone Vector DB

This notebook reconstructs customer support conversation threads from Twitter support datasets () and inserts them into a Pinecone vector database using Pinecone Integrated Hosted Inference ().

- **Record ID ()**:  of the initial starting message from the user.
- **Text ()**: Combined customer messages starting from initial issue up to the first brand reply.
- **Metadata ()**: The text of the very first reply from .

In [ ]:
# Step 1: Install Dependencies in Google Colab
!pip install -q pinecone-client pandas numpy python-dotenv

In [ ]:
# Step 2: Configure Environment Variables & Pinecone API Key
import os
import getpass

# Try loading from Google Colab Secrets (userdata) or prompt for key
if "PINECONE_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
    except Exception:
        os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API Key: ")

os.environ["PINECONE_INDEX_NAME"] = "customer-support-resolutions"
os.environ["PINECONE_NAMESPACE"] = "__default__"
print("✅ Environment configured successfully!")

In [ ]:
# Step 3: Define Data Ingestion & Thread Reconstruction Pipeline
from typing import List, Dict, Any, Optional
import pandas as pd
import numpy as np
from pinecone import Pinecone

def load_tweets_from_csv(csv_path: str) -> pd.DataFrame:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")
    print(f"📂 Reading CSV dataset from: {csv_path}...")
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    print(f"✅ Loaded {len(df):,} raw tweets.")
    return df

def filter_applesupport_tweets(df: pd.DataFrame, target_brand: str = "AppleSupport") -> pd.DataFrame:
    brand_lower = target_brand.lower()
    author_series = df["author_id"].str.lower()
    text_series = df["text"].str.lower()
    is_author = author_series == brand_lower
    is_mentioned = text_series.str.contains(f"@{brand_lower}", regex=False, na=False)
    mask = np.logical_or(is_author, is_mentioned)
    filtered_df = df[mask].copy()
    filtered_df["is_brand_author"] = is_author[mask]
    filtered_df["is_inbound"] = filtered_df["inbound"].str.lower() == "true"
    print(f"✅ Isolated {len(filtered_df):,} '{target_brand}' related tweets.")
    return filtered_df

def reconstruct_user_brand_pairs(df: pd.DataFrame, target_brand: str = "AppleSupport") -> List[Dict[str, Any]]:
    brand_lower = target_brand.lower()
    tweet_map = {row["tweet_id"]: row for row in df.to_dict("records")}
    root_user_tweets = []
    for tid, t in tweet_map.items():
        if t["author_id"].lower() != brand_lower and (f"@{brand_lower}" in t["text"].lower()):
            parent_id = t.get("in_response_to_tweet_id", "").strip()
            if not parent_id or parent_id not in tweet_map:
                root_user_tweets.append(t)
    records = []
    for root in root_user_tweets:
        user_messages = [root]
        first_brand_reply = None
        queue = [root["tweet_id"]]
        visited = set()
        while queue and not first_brand_reply:
            curr_id = queue.pop(0)
            if curr_id in visited or curr_id not in tweet_map:
                continue
            visited.add(curr_id)
            t = tweet_map[curr_id]
            if t["author_id"].lower() == brand_lower:
                first_brand_reply = t
                break
            elif curr_id != root["tweet_id"]:
                user_messages.append(t)
            resp_ids_str = t.get("response_tweet_id", "").strip()
            if resp_ids_str:
                for r_id in resp_ids_str.split(","):
                    r_id = r_id.strip()
                    if r_id in tweet_map and r_id not in visited:
                        queue.append(r_id)
        if first_brand_reply:
            user_text_lines = [f"Customer ({t['author_id']}): {t['text']}" for t in user_messages]
            user_text_combined = "
".join(user_text_lines)
            records.append({
                "_id": root["tweet_id"],
                "text": user_text_combined,
                "brand_reply": first_brand_reply["text"],
                "brand": target_brand,
                "reply_tweet_id": first_brand_reply["tweet_id"],
                "created_at": root.get("created_at", ""),
                "user_tweet_count": len(user_messages)
            })
    print(f"✅ Extracted {len(records):,} first-user to first-brand conversation records.")
    return records

def upsert_records_to_pinecone(records: List[Dict[str, Any]], batch_size: int = 100) -> int:
    if not records:
        print("⚠️ No records to upsert.")
        return 0
    api_key = os.getenv("PINECONE_API_KEY")
    index_name = os.getenv("PINECONE_INDEX_NAME", "customer-support-resolutions")
    namespace = os.getenv("PINECONE_NAMESPACE", "__default__")
    pc = Pinecone(api_key=api_key)
    index = pc.Index(index_name)
    total_records = len(records)
    total_batches = (total_records + batch_size - 1) // batch_size
    print(f"
🌲 Connecting to Pinecone Index '{index_name}'...")
    print(f"🚀 Upserting {total_records:,} records in {total_batches:,} batches...")
    upserted_count = 0
    for i in range(0, total_records, batch_size):
        batch = records[i : i + batch_size]
        index.upsert_records(namespace=namespace, records=batch)
        upserted_count += len(batch)
        print(f"  └─ Batch {i//batch_size + 1}/{total_batches}: Upserted {upserted_count:,}/{total_records:,} records.")
    print(f"✅ Successfully inserted {upserted_count:,} records into Pinecone DB.")
    return upserted_count

In [ ]:
# Step 4: Run Data Ingestion & Thread Reconstruction
CSV_FILE_PATH = "twcs.csv"  # Path to CSV file uploaded to Colab or Google Drive

df = load_tweets_from_csv(CSV_FILE_PATH)
filtered_df = filter_applesupport_tweets(df, target_brand="AppleSupport")
records = reconstruct_user_brand_pairs(df, target_brand="AppleSupport")

print("
--- Preview Top Records ---")
for i, record in enumerate(records[:2]):
    print(f"
[Record #{i+1}] ID: {record['_id']} | Reply Tweet ID: {record['reply_tweet_id']}")
    print("Text Field (User Messages):")
    print(record["text"])
    print("Brand Reply Metadata:")
    print(record["brand_reply"])

In [ ]:
# Step 5: Upsert Records to Pinecone DB
# Set UPSERT_LIMIT to None to upsert all 51,340 records, or integer (e.g. 20) for a subset
UPSERT_LIMIT = None

records_to_upsert = records if UPSERT_LIMIT is None else records[:UPSERT_LIMIT]
upsert_records_to_pinecone(records_to_upsert, batch_size=100)